# Hyperspectral analysis and prototype land-use scoring

This notebook was developed as part of the **FLUXE Makeathlon 2026** challenge submission.
It explores EnMAP hyperspectral scenes, computes vegetation/water/soil indices,
runs a **StandardScaler → PCA → K-Means** pixel segmentation experiment,
and generates prototype suitability scores for five possible land uses.

> **Reproducibility & data disclosure.** Original satellite scenes, metadata XML,
> PVGIS CSVs, wind spreadsheets, and `infrastructure_synthetic_indicators.csv`
> are **not included** in this repository. The full notebook cannot run
> end-to-end without those inputs. See `ml/README.md` for details.

> **Prototype caveat.** Some infrastructure and terrain inputs are synthetic,
> and one late-stage heuristic deliberately restricts some battery/data-center
> scores relative to wind for a demo narrative. Scores are not calibrated
> predictions or validated investment recommendations.

To run in Colab, mount Drive and set `MAKEATHON_DATA_DIR` (optional), or
edit the input directory in the following cell.


In [ ]:
import os
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and os.path.isdir("/content"):
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path

DATA_DIR = Path(os.environ.get("MAKEATHON_DATA_DIR", "/content/drive/MyDrive/images_makeathlon"))

print("DATA_DIR exists:", DATA_DIR.exists())
print("DATA_DIR:", DATA_DIR)

In [ ]:
for item in sorted(DATA_DIR.iterdir()):
    if item.is_dir():
        print("[DIR] ", item.name)
    else:
        size_mb = item.stat().st_size / (1024 * 1024)
        print("[FILE]", item.name, f"| {size_mb:.2f} MB")

In [ ]:
from pathlib import Path

tif_files = sorted(
    list(DATA_DIR.rglob("*.tif")) +
    list(DATA_DIR.rglob("*.tiff")) +
    list(DATA_DIR.rglob("*.TIF")) +
    list(DATA_DIR.rglob("*.TIFF"))
)

print("Number of tif files found:", len(tif_files))
print()

for i, path in enumerate(tif_files):
    size_mb = path.stat().st_size / (1024 * 1024)
    relative_path = path.relative_to(DATA_DIR)
    print(f"{i}: {relative_path}")
    print(f"   size: {size_mb:.2f} MB")

In [ ]:
spectral_files = [p for p in tif_files if p.name == "SPECTRAL_IMAGE.TIF"]

print("Number of spectral image files:", len(spectral_files))
print()

for i, path in enumerate(spectral_files):
    print(f"{i}: {path.relative_to(DATA_DIR)}")

In [ ]:
!pip install rasterio matplotlib numpy pandas

In [ ]:
import rasterio
import pandas as pd

rows = []

for path in spectral_files:
    try:
        with rasterio.open(path) as src:
            rows.append({
                "folder": path.parent.name,
                "file": path.name,
                "bands": src.count,
                "width": src.width,
                "height": src.height,
                "crs": str(src.crs),
                "dtype": src.dtypes[0],
                "nodata": src.nodata,
                "driver": src.driver,
                "bounds": src.bounds,
                "transform": src.transform,
                "size_mb": path.stat().st_size / (1024 * 1024),
            })
    except Exception as e:
        rows.append({
            "folder": path.parent.name,
            "file": path.name,
            "error": str(e)
        })

spectral_metadata_df = pd.DataFrame(rows)
spectral_metadata_df

In [ ]:
spectral_metadata_df[[
    "folder",
    "bands",
    "width",
    "height",
    "crs",
    "dtype",
    "nodata",
    "size_mb"
]]

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Πάρε το πρώτο spectral image για δοκιμή
test_path = spectral_files[0]

print("Testing file:", test_path.relative_to(DATA_DIR))

with rasterio.open(test_path) as src:
    cube = src.read()  # shape: bands, height, width
    profile = src.profile
    nodata = src.nodata

print("Raw cube shape:", cube.shape)
print("Nodata value:", nodata)
print("Dtype:", cube.dtype)
print("Min before cleaning:", np.nanmin(cube))
print("Max before cleaning:", np.nanmax(cube))
print("Mean before cleaning:", np.nanmean(cube))

In [ ]:
cube = cube.astype(np.float32)

if nodata is not None:
    cube[cube == nodata] = np.nan

print("Min after cleaning:", np.nanmin(cube))
print("Max after cleaning:", np.nanmax(cube))
print("Mean after cleaning:", np.nanmean(cube))
print("Number of NaN values:", np.isnan(cube).sum())
print("Total values:", cube.size)

In [ ]:
cube_hwc = np.moveaxis(cube, 0, -1)

print("Cube HWC shape:", cube_hwc.shape)

In [ ]:
h, w, b = cube_hwc.shape

center_y = h // 2
center_x = w // 2

spectrum = cube_hwc[center_y, center_x, :]

plt.figure(figsize=(10, 4))
plt.plot(spectrum)
plt.title("Spectral signature of center pixel")
plt.xlabel("Band index")
plt.ylabel("Reflectance / value")
plt.grid(True)
plt.show()

print("Center pixel location:", center_y, center_x)
print("Spectrum shape:", spectrum.shape)
print("Spectrum min:", np.nanmin(spectrum))
print("Spectrum max:", np.nanmax(spectrum))
print("Spectrum mean:", np.nanmean(spectrum))

In [ ]:
mean_spectrum = np.nanmean(cube_hwc, axis=(0, 1))

plt.figure(figsize=(10, 4))
plt.plot(mean_spectrum)
plt.title("Mean spectral signature of plot")
plt.xlabel("Band index")
plt.ylabel("Mean reflectance / value")
plt.grid(True)
plt.show()

print("Mean spectrum shape:", mean_spectrum.shape)
print("Mean spectrum min:", np.nanmin(mean_spectrum))
print("Mean spectrum max:", np.nanmax(mean_spectrum))

In [ ]:
nan_per_band = np.isnan(cube_hwc).sum(axis=(0, 1))
total_pixels = cube_hwc.shape[0] * cube_hwc.shape[1]
nan_ratio_per_band = nan_per_band / total_pixels

print("Total pixels per band:", total_pixels)
print()

for i, ratio in enumerate(nan_ratio_per_band):
    if ratio > 0:
        print(f"Band {i}: NaN ratio = {ratio:.2%}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(nan_ratio_per_band)
plt.title("NaN ratio per spectral band")
plt.xlabel("Band index")
plt.ylabel("NaN ratio")
plt.grid(True)
plt.show()

In [ ]:
with rasterio.open(test_path) as src:
    print("General tags:")
    print(src.tags())
    print()

    print("Band 1 tags:")
    print(src.tags(1))
    print()

    print("Band 50 tags:")
    print(src.tags(50))
    print()

    print("Band 100 tags:")
    print(src.tags(100))
    print()

    print("Band 150 tags:")
    print(src.tags(150))
    print()

    print("Band 224 tags:")
    print(src.tags(224))

In [ ]:
all_files = sorted(DATA_DIR.rglob("*"))

for path in all_files:
    if path.is_file() and path.suffix.lower() not in [".tif", ".tiff"]:
        size_kb = path.stat().st_size / 1024
        print(path.relative_to(DATA_DIR), f"| {size_kb:.1f} KB")

In [ ]:
metadata_path = test_path.parent / "METADATA.XML"

print("Metadata path:", metadata_path.relative_to(DATA_DIR))
print("Exists:", metadata_path.exists())
print("Size MB:", metadata_path.stat().st_size / (1024 * 1024))

In [ ]:
text = metadata_path.read_text(errors="ignore")

keywords = [
    "wavelength",
    "WAVELENGTH",
    "center",
    "CENTER",
    "band",
    "BAND",
    "FWHM",
    "fwhm"
]

for kw in keywords:
    print(kw, "=>", text.lower().count(kw.lower()))

In [ ]:
def print_context(text, keyword, n=500, max_hits=5):
    text_lower = text.lower()
    keyword_lower = keyword.lower()

    start = 0
    hits = 0

    while hits < max_hits:
        idx = text_lower.find(keyword_lower, start)
        if idx == -1:
            break

        a = max(0, idx - n)
        b = min(len(text), idx + n)

        print("=" * 80)
        print(text[a:b])
        print("=" * 80)

        start = idx + len(keyword)
        hits += 1

print_context(text, "wavelength", n=700, max_hits=5)

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

metadata_path = test_path.parent / "METADATA.XML"

tree = ET.parse(metadata_path)
root = tree.getroot()

bands_info = []

# Ψάχνουμε όλα τα bandID tags, όπου κι αν βρίσκονται στο XML
for band in root.iter():
    if band.tag.endswith("bandID"):
        band_number = band.attrib.get("number")

        wavelength = None
        fwhm = None
        gain = None
        offset = None

        for child in band:
            tag = child.tag.split("}")[-1]  # handles XML namespaces if they exist
            text_value = child.text

            if tag == "wavelengthCenterOfBand":
                wavelength = float(text_value)
            elif tag == "FWHMOfBand":
                fwhm = float(text_value)
            elif tag == "GainOfBand":
                gain = float(text_value)
            elif tag == "OffsetOfBand":
                offset = float(text_value)

        if band_number is not None and wavelength is not None:
            bands_info.append({
                "band_number_xml": int(band_number),
                "band_index_python": int(band_number) - 1,
                "wavelength_nm": wavelength,
                "fwhm": fwhm,
                "gain": gain,
                "offset": offset,
            })

bands_df = pd.DataFrame(bands_info)

print("Number of parsed bands:", len(bands_df))
bands_df.head()

In [ ]:
display(bands_df.head(10))
display(bands_df.tail(10))

In [ ]:
bad_band_indices = [130, 131, 132, 133, 134]

bands_df[bands_df["band_index_python"].isin(bad_band_indices)]

In [ ]:
import numpy as np

def closest_band(target_wavelength_nm, bands_df, avoid_bad=True, bad_band_indices=None):
    df = bands_df.copy()

    if avoid_bad and bad_band_indices is not None:
        df = df[~df["band_index_python"].isin(bad_band_indices)]

    idx = (df["wavelength_nm"] - target_wavelength_nm).abs().idxmin()
    row = df.loc[idx]

    return {
        "target_nm": target_wavelength_nm,
        "band_number_xml": int(row["band_number_xml"]),
        "band_index_python": int(row["band_index_python"]),
        "actual_wavelength_nm": float(row["wavelength_nm"]),
        "difference_nm": float(abs(row["wavelength_nm"] - target_wavelength_nm)),
    }

In [ ]:
targets = {
    "blue": 490,
    "green": 560,
    "red": 665,
    "red_edge": 720,
    "nir": 842,
    "swir1": 1610,
    "swir2": 2200,
}

selected_bands = []

for name, target in targets.items():
    result = closest_band(
        target_wavelength_nm=target,
        bands_df=bands_df,
        avoid_bad=True,
        bad_band_indices=bad_band_indices
    )
    result["name"] = name
    selected_bands.append(result)

selected_bands_df = pd.DataFrame(selected_bands)
selected_bands_df[[
    "name",
    "target_nm",
    "band_index_python",
    "band_number_xml",
    "actual_wavelength_nm",
    "difference_nm"
]]

In [ ]:
def get_selected_band_index(name, selected_bands_df):
    return int(selected_bands_df.loc[selected_bands_df["name"] == name, "band_index_python"].iloc[0])

blue_idx = get_selected_band_index("blue", selected_bands_df)
green_idx = get_selected_band_index("green", selected_bands_df)
red_idx = get_selected_band_index("red", selected_bands_df)
red_edge_idx = get_selected_band_index("red_edge", selected_bands_df)
nir_idx = get_selected_band_index("nir", selected_bands_df)
swir1_idx = get_selected_band_index("swir1", selected_bands_df)
swir2_idx = get_selected_band_index("swir2", selected_bands_df)

print("blue_idx:", blue_idx)
print("green_idx:", green_idx)
print("red_idx:", red_idx)
print("red_edge_idx:", red_edge_idx)
print("nir_idx:", nir_idx)
print("swir1_idx:", swir1_idx)
print("swir2_idx:", swir2_idx)

In [ ]:
def normalized_difference(a, b):
    return (a - b) / (a + b + 1e-6)

In [ ]:
red = cube_hwc[:, :, red_idx]
nir = cube_hwc[:, :, nir_idx]

ndvi = normalized_difference(nir, red)

print("NDVI shape:", ndvi.shape)
print("NDVI min:", np.nanmin(ndvi))
print("NDVI max:", np.nanmax(ndvi))
print("NDVI mean:", np.nanmean(ndvi))
print("NDVI median:", np.nanmedian(ndvi))

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(ndvi, vmin=-1, vmax=1)
plt.colorbar(label="NDVI")
plt.title("NDVI heatmap - first plot")
plt.axis("off")
plt.show()

In [ ]:
valid_ndvi = ndvi[~np.isnan(ndvi)]

total_valid = len(valid_ndvi)

low_veg = np.sum(valid_ndvi < 0.2)
medium_veg = np.sum((valid_ndvi >= 0.2) & (valid_ndvi < 0.5))
high_veg = np.sum(valid_ndvi >= 0.5)

print("Valid NDVI pixels:", total_valid)
print("Low vegetation pixels (<0.2):", low_veg, f"({low_veg / total_valid:.1%})")
print("Medium vegetation pixels (0.2-0.5):", medium_veg, f"({medium_veg / total_valid:.1%})")
print("High vegetation pixels (>=0.5):", high_veg, f"({high_veg / total_valid:.1%})")

In [ ]:
red_band = cube_hwc[:, :, red_idx]
green_band = cube_hwc[:, :, green_idx]
blue_band = cube_hwc[:, :, blue_idx]

rgb = np.dstack([red_band, green_band, blue_band])

print("RGB shape:", rgb.shape)
print("RGB min:", np.nanmin(rgb))
print("RGB max:", np.nanmax(rgb))
print("RGB mean:", np.nanmean(rgb))

In [ ]:
def normalize_for_display(img, lower=2, upper=98):
    img_norm = img.copy().astype(np.float32)

    for c in range(img_norm.shape[2]):
        band = img_norm[:, :, c]

        p_low = np.nanpercentile(band, lower)
        p_high = np.nanpercentile(band, upper)

        img_norm[:, :, c] = (band - p_low) / (p_high - p_low + 1e-6)
        img_norm[:, :, c] = np.clip(img_norm[:, :, c], 0, 1)

    # Τα NaN για εμφάνιση τα κάνουμε 0
    img_norm = np.nan_to_num(img_norm, nan=0.0)

    return img_norm

rgb_norm = normalize_for_display(rgb)

print("RGB normalized min:", rgb_norm.min())
print("RGB normalized max:", rgb_norm.max())

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(rgb_norm)
plt.title("RGB preview - first plot")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(rgb_norm, interpolation="nearest")
plt.title("RGB preview - enlarged")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import xml.etree.ElementTree as ET
from pathlib import Path

def parse_bands_metadata(metadata_path):
    """
    Reads METADATA.XML and returns a dataframe with:
    band_number_xml, band_index_python, wavelength_nm, fwhm, gain, offset
    """
    tree = ET.parse(metadata_path)
    root = tree.getroot()

    bands_info = []

    for band in root.iter():
        if band.tag.endswith("bandID"):
            band_number = band.attrib.get("number")

            wavelength = None
            fwhm = None
            gain = None
            offset = None

            for child in band:
                tag = child.tag.split("}")[-1]
                text_value = child.text

                if tag == "wavelengthCenterOfBand":
                    wavelength = float(text_value)
                elif tag == "FWHMOfBand":
                    fwhm = float(text_value)
                elif tag == "GainOfBand":
                    gain = float(text_value)
                elif tag == "OffsetOfBand":
                    offset = float(text_value)

            if band_number is not None and wavelength is not None:
                bands_info.append({
                    "band_number_xml": int(band_number),
                    "band_index_python": int(band_number) - 1,
                    "wavelength_nm": wavelength,
                    "fwhm": fwhm,
                    "gain": gain,
                    "offset": offset,
                })

    return pd.DataFrame(bands_info)


def load_hyperspectral_cube(spectral_path):
    """
    Loads SPECTRAL_IMAGE.TIF and returns cube in HWC format:
    height, width, bands
    """
    with rasterio.open(spectral_path) as src:
        cube = src.read().astype(np.float32)  # bands, height, width
        nodata = src.nodata
        profile = src.profile

    if nodata is not None:
        cube[cube == nodata] = np.nan

    cube_hwc = np.moveaxis(cube, 0, -1)  # height, width, bands

    return cube_hwc, profile, nodata


def get_bad_bands(cube_hwc, threshold=0.5):
    """
    Returns band indices where more than threshold percentage of pixels are NaN.
    """
    h, w, b = cube_hwc.shape
    total_pixels = h * w

    nan_per_band = np.isnan(cube_hwc).sum(axis=(0, 1))
    nan_ratio_per_band = nan_per_band / total_pixels

    bad_band_indices = np.where(nan_ratio_per_band > threshold)[0].tolist()

    return bad_band_indices, nan_ratio_per_band


def closest_band(target_wavelength_nm, bands_df, bad_band_indices=None):
    """
    Finds closest available band to target wavelength, avoiding bad bands.
    """
    df = bands_df.copy()

    if bad_band_indices is not None:
        df = df[~df["band_index_python"].isin(bad_band_indices)]

    idx = (df["wavelength_nm"] - target_wavelength_nm).abs().idxmin()
    row = df.loc[idx]

    return int(row["band_index_python"])


def normalized_difference(a, b):
    """
    Generic normalized difference index.
    """
    return (a - b) / (a + b + 1e-6)


def safe_stats(arr, prefix):
    """
    Computes summary stats for one index map.
    """
    valid = arr[~np.isnan(arr)]

    if len(valid) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_valid_pixels": 0,
        }

    return {
        f"{prefix}_mean": float(np.nanmean(arr)),
        f"{prefix}_median": float(np.nanmedian(arr)),
        f"{prefix}_std": float(np.nanstd(arr)),
        f"{prefix}_min": float(np.nanmin(arr)),
        f"{prefix}_max": float(np.nanmax(arr)),
        f"{prefix}_valid_pixels": int(len(valid)),
    }

In [ ]:
def compute_indices_for_plot(spectral_path):
    """
    Computes spectral indices for one plot/folder.
    Returns:
    - summary_row: dictionary with stats
    - index_maps: dictionary with full 2D index arrays
    """
    plot_name = spectral_path.parent.name
    metadata_path = spectral_path.parent / "METADATA.XML"

    cube_hwc, profile, nodata = load_hyperspectral_cube(spectral_path)
    bands_df = parse_bands_metadata(metadata_path)

    bad_band_indices, nan_ratio_per_band = get_bad_bands(cube_hwc, threshold=0.5)

    # Target wavelengths in nm
    blue_idx = closest_band(490, bands_df, bad_band_indices)
    green_idx = closest_band(560, bands_df, bad_band_indices)
    red_idx = closest_band(665, bands_df, bad_band_indices)
    red_edge_idx = closest_band(720, bands_df, bad_band_indices)
    nir_idx = closest_band(842, bands_df, bad_band_indices)
    swir1_idx = closest_band(1610, bands_df, bad_band_indices)
    swir2_idx = closest_band(2200, bands_df, bad_band_indices)

    blue = cube_hwc[:, :, blue_idx]
    green = cube_hwc[:, :, green_idx]
    red = cube_hwc[:, :, red_idx]
    red_edge = cube_hwc[:, :, red_edge_idx]
    nir = cube_hwc[:, :, nir_idx]
    swir1 = cube_hwc[:, :, swir1_idx]
    swir2 = cube_hwc[:, :, swir2_idx]

    # Spectral indices
    ndvi = normalized_difference(nir, red)
    ndwi = normalized_difference(green, nir)
    ndmi = normalized_difference(nir, swir1)
    ndre = normalized_difference(nir, red_edge)
    nbr = normalized_difference(nir, swir2)
    gndvi = normalized_difference(nir, green)

    # SAVI with L = 0.5
    L = 0.5
    savi = ((nir - red) / (nir + red + L + 1e-6)) * (1 + L)

    index_maps = {
        "NDVI": ndvi,
        "NDWI": ndwi,
        "NDMI": ndmi,
        "NDRE": ndre,
        "NBR": nbr,
        "GNDVI": gndvi,
        "SAVI": savi,
    }

    summary_row = {
        "plot_name": plot_name,
        "spectral_file": str(spectral_path.relative_to(DATA_DIR)),
        "height": cube_hwc.shape[0],
        "width": cube_hwc.shape[1],
        "bands": cube_hwc.shape[2],
        "bad_bands_count": len(bad_band_indices),
        "bad_bands": str(bad_band_indices),

        "blue_band_idx": blue_idx,
        "green_band_idx": green_idx,
        "red_band_idx": red_idx,
        "red_edge_band_idx": red_edge_idx,
        "nir_band_idx": nir_idx,
        "swir1_band_idx": swir1_idx,
        "swir2_band_idx": swir2_idx,

        "blue_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == blue_idx, "wavelength_nm"].iloc[0]),
        "green_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == green_idx, "wavelength_nm"].iloc[0]),
        "red_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == red_idx, "wavelength_nm"].iloc[0]),
        "red_edge_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == red_edge_idx, "wavelength_nm"].iloc[0]),
        "nir_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == nir_idx, "wavelength_nm"].iloc[0]),
        "swir1_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == swir1_idx, "wavelength_nm"].iloc[0]),
        "swir2_wavelength_nm": float(bands_df.loc[bands_df["band_index_python"] == swir2_idx, "wavelength_nm"].iloc[0]),
    }

    for index_name, index_arr in index_maps.items():
        summary_row.update(safe_stats(index_arr, index_name))

    # Extra NDVI vegetation percentages
    valid_ndvi = ndvi[~np.isnan(ndvi)]
    if len(valid_ndvi) > 0:
        summary_row["NDVI_low_veg_pct"] = float(np.sum(valid_ndvi < 0.2) / len(valid_ndvi))
        summary_row["NDVI_medium_veg_pct"] = float(np.sum((valid_ndvi >= 0.2) & (valid_ndvi < 0.5)) / len(valid_ndvi))
        summary_row["NDVI_high_veg_pct"] = float(np.sum(valid_ndvi >= 0.5) / len(valid_ndvi))
    else:
        summary_row["NDVI_low_veg_pct"] = np.nan
        summary_row["NDVI_medium_veg_pct"] = np.nan
        summary_row["NDVI_high_veg_pct"] = np.nan

    return summary_row, index_maps

In [ ]:
all_summary_rows = []
all_index_maps = {}

for spectral_path in spectral_files:
    print("=" * 80)
    print("Processing:", spectral_path.relative_to(DATA_DIR))

    summary_row, index_maps = compute_indices_for_plot(spectral_path)

    all_summary_rows.append(summary_row)
    all_index_maps[summary_row["plot_name"]] = index_maps

    print("Done:", summary_row["plot_name"])
    print("NDVI mean:", summary_row["NDVI_mean"])
    print("NDWI mean:", summary_row["NDWI_mean"])
    print("NDMI mean:", summary_row["NDMI_mean"])

indices_summary_df = pd.DataFrame(all_summary_rows)

indices_summary_df

In [ ]:
main_cols = [
    "plot_name",
    "height",
    "width",
    "bands",

    "NDVI_mean",
    "NDVI_median",
    "NDVI_min",
    "NDVI_max",
    "NDVI_high_veg_pct",

    "NDWI_mean",
    "NDMI_mean",
    "NDRE_mean",
    "NBR_mean",
    "GNDVI_mean",
    "SAVI_mean",
]

indices_summary_df[main_cols]

In [ ]:
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

csv_path = OUTPUT_DIR / "plot_spectral_indices_summary.csv"

indices_summary_df.to_csv(csv_path, index=False)

print("Saved CSV to:")
print(csv_path)

In [ ]:
clean_csv_path = OUTPUT_DIR / "plot_spectral_indices_clean.csv"

indices_summary_df[main_cols].to_csv(clean_csv_path, index=False)

print("Saved clean CSV to:")
print(clean_csv_path)

In [ ]:
indices_summary_df[main_cols]

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pathlib import Path

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def parse_bands_metadata(metadata_path):
    tree = ET.parse(metadata_path)
    root = tree.getroot()

    bands_info = []

    for band in root.iter():
        if band.tag.endswith("bandID"):
            band_number = band.attrib.get("number")

            wavelength = None
            fwhm = None
            gain = None
            offset = None

            for child in band:
                tag = child.tag.split("}")[-1]
                text_value = child.text

                if tag == "wavelengthCenterOfBand":
                    wavelength = float(text_value)
                elif tag == "FWHMOfBand":
                    fwhm = float(text_value)
                elif tag == "GainOfBand":
                    gain = float(text_value)
                elif tag == "OffsetOfBand":
                    offset = float(text_value)

            if band_number is not None and wavelength is not None:
                bands_info.append({
                    "band_number_xml": int(band_number),
                    "band_index_python": int(band_number) - 1,
                    "wavelength_nm": wavelength,
                    "fwhm": fwhm,
                    "gain": gain,
                    "offset": offset,
                })

    return pd.DataFrame(bands_info)


def load_hyperspectral_cube(spectral_path):
    with rasterio.open(spectral_path) as src:
        cube = src.read().astype(np.float32)  # bands, h, w
        nodata = src.nodata
        profile = src.profile

    if nodata is not None:
        cube[cube == nodata] = np.nan

    cube_hwc = np.moveaxis(cube, 0, -1)  # h, w, bands
    return cube_hwc, profile, nodata


def get_bad_bands(cube_hwc, threshold=0.5):
    h, w, b = cube_hwc.shape
    total_pixels = h * w

    nan_per_band = np.isnan(cube_hwc).sum(axis=(0, 1))
    nan_ratio_per_band = nan_per_band / total_pixels

    bad_band_indices = np.where(nan_ratio_per_band > threshold)[0].tolist()
    return bad_band_indices, nan_ratio_per_band


def closest_band(target_wavelength_nm, bands_df, bad_band_indices=None):
    df = bands_df.copy()
    if bad_band_indices is not None:
        df = df[~df["band_index_python"].isin(bad_band_indices)]

    idx = (df["wavelength_nm"] - target_wavelength_nm).abs().idxmin()
    row = df.loc[idx]
    return int(row["band_index_python"])


def normalized_difference(a, b):
    return (a - b) / (a + b + 1e-6)


def safe_stats(arr, prefix):
    valid = arr[~np.isnan(arr)]

    if len(valid) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_valid_pixels": 0,
        }

    return {
        f"{prefix}_mean": float(np.nanmean(arr)),
        f"{prefix}_median": float(np.nanmedian(arr)),
        f"{prefix}_std": float(np.nanstd(arr)),
        f"{prefix}_min": float(np.nanmin(arr)),
        f"{prefix}_max": float(np.nanmax(arr)),
        f"{prefix}_valid_pixels": int(len(valid)),
    }


def normalize_for_display(img, lower=2, upper=98):
    img_norm = img.copy().astype(np.float32)

    for c in range(img_norm.shape[2]):
        band = img_norm[:, :, c]
        p_low = np.nanpercentile(band, lower)
        p_high = np.nanpercentile(band, upper)

        img_norm[:, :, c] = (band - p_low) / (p_high - p_low + 1e-6)
        img_norm[:, :, c] = np.clip(img_norm[:, :, c], 0, 1)

    img_norm = np.nan_to_num(img_norm, nan=0.0)
    return img_norm

In [ ]:
def compute_indices_for_plot_v2(spectral_path):
    plot_name = spectral_path.parent.name
    metadata_path = spectral_path.parent / "METADATA.XML"

    cube_hwc, profile, nodata = load_hyperspectral_cube(spectral_path)
    bands_df = parse_bands_metadata(metadata_path)

    bad_band_indices, nan_ratio_per_band = get_bad_bands(cube_hwc, threshold=0.5)

    # Core bands
    targets = {
        "blue": 490,
        "green": 560,
        "red": 665,
        "red_edge": 720,
        "nir": 842,
        "swir1": 1610,
        "swir2": 2200,

        # extra hyperspectral bands
        "b531": 531,
        "b570": 570,
        "r681": 681,
        "r709": 709,
        "r754": 754,
    }

    selected = {}
    for name, target in targets.items():
        selected[name] = closest_band(target, bands_df, bad_band_indices)

    # Extract bands
    blue = cube_hwc[:, :, selected["blue"]]
    green = cube_hwc[:, :, selected["green"]]
    red = cube_hwc[:, :, selected["red"]]
    red_edge = cube_hwc[:, :, selected["red_edge"]]
    nir = cube_hwc[:, :, selected["nir"]]
    swir1 = cube_hwc[:, :, selected["swir1"]]
    swir2 = cube_hwc[:, :, selected["swir2"]]

    b531 = cube_hwc[:, :, selected["b531"]]
    b570 = cube_hwc[:, :, selected["b570"]]
    r681 = cube_hwc[:, :, selected["r681"]]
    r709 = cube_hwc[:, :, selected["r709"]]
    r754 = cube_hwc[:, :, selected["r754"]]

    # Indices
    ndvi = normalized_difference(nir, red)
    ndwi = normalized_difference(green, nir)
    ndmi = normalized_difference(nir, swir1)
    ndre = normalized_difference(nir, red_edge)
    nbr = normalized_difference(nir, swir2)
    gndvi = normalized_difference(nir, green)

    # New indices
    msi = swir1 / (nir + 1e-6)                       # higher = more stress/drier
    cire = (nir / (red_edge + 1e-6)) - 1            # chlorophyll index red edge
    gci = (nir / (green + 1e-6)) - 1                # green chlorophyll index
    psri = (red - blue) / (red_edge + 1e-6)         # senescence / stress
    bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + 1e-6)
    pri = normalized_difference(b531, b570)         # photosynthetic reflectance index
    mtci = (r754 - r709) / (r709 - r681 + 1e-6)     # chlorophyll-related

    index_maps = {
        "NDVI": ndvi,
        "NDWI": ndwi,
        "NDMI": ndmi,
        "NDRE": ndre,
        "NBR": nbr,
        "GNDVI": gndvi,
        "MSI": msi,
        "CIRE": cire,
        "GCI": gci,
        "PSRI": psri,
        "BSI": bsi,
        "PRI": pri,
        "MTCI": mtci,
    }

    # RGB
    rgb = np.dstack([red, green, blue])
    rgb_norm = normalize_for_display(rgb)

    summary_row = {
        "plot_name": plot_name,
        "spectral_file": str(spectral_path.relative_to(DATA_DIR)),
        "height": cube_hwc.shape[0],
        "width": cube_hwc.shape[1],
        "bands": cube_hwc.shape[2],
        "bad_bands_count": len(bad_band_indices),
        "bad_bands": str(bad_band_indices),
    }

    # save selected wavelengths/bands
    for name, idx in selected.items():
        summary_row[f"{name}_band_idx"] = idx
        summary_row[f"{name}_wavelength_nm"] = float(
            bands_df.loc[bands_df["band_index_python"] == idx, "wavelength_nm"].iloc[0]
        )

    # stats per index
    for index_name, arr in index_maps.items():
        summary_row.update(safe_stats(arr, index_name))

    return summary_row, index_maps, rgb_norm, cube_hwc, selected

In [ ]:
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

VIS_DIR = OUTPUT_DIR / "visualizations"
VIS_DIR.mkdir(exist_ok=True)

all_summary_rows = []
analysis_results = {}

for spectral_path in spectral_files:
    print("=" * 80)
    print("Processing:", spectral_path.relative_to(DATA_DIR))

    summary_row, index_maps, rgb_norm, cube_hwc, selected = compute_indices_for_plot_v2(spectral_path)

    all_summary_rows.append(summary_row)
    analysis_results[summary_row["plot_name"]] = {
        "summary": summary_row,
        "index_maps": index_maps,
        "rgb_norm": rgb_norm,
        "cube_hwc": cube_hwc,
        "selected": selected,
        "spectral_path": spectral_path,
    }

    print("Done:", summary_row["plot_name"])

indices_summary_v2_df = pd.DataFrame(all_summary_rows)

indices_summary_v2_df.head()

In [ ]:
csv_v2_path = OUTPUT_DIR / "plot_spectral_indices_extended.csv"
indices_summary_v2_df.to_csv(csv_v2_path, index=False)

print("Saved extended CSV to:")
print(csv_v2_path)

In [ ]:
def save_plot_dashboard(plot_name, result, out_dir):
    rgb = result["rgb_norm"]
    index_maps = result["index_maps"]

    panels = [
        ("RGB", rgb),
        ("NDVI", index_maps["NDVI"]),
        ("NDWI", index_maps["NDWI"]),
        ("NDMI", index_maps["NDMI"]),
        ("NDRE", index_maps["NDRE"]),
        ("NBR", index_maps["NBR"]),
        ("GNDVI", index_maps["GNDVI"]),
        ("MSI", index_maps["MSI"]),
        ("CIRE", index_maps["CIRE"]),
        ("GCI", index_maps["GCI"]),
        ("BSI", index_maps["BSI"]),
        ("PSRI", index_maps["PSRI"]),
    ]

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()

    for ax, (title, data) in zip(axes, panels):
        if title == "RGB":
            ax.imshow(data, interpolation="nearest")
        else:
            im = ax.imshow(data, interpolation="nearest")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        ax.set_title(title)
        ax.axis("off")

    # Αν περισσεύουν axes
    for ax in axes[len(panels):]:
        ax.axis("off")

    plt.suptitle(plot_name, fontsize=16)
    plt.tight_layout()

    save_path = out_dir / f"{plot_name}_dashboard.png"
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

    return save_path

In [ ]:
dashboard_paths = []

for plot_name, result in analysis_results.items():
    print("Saving dashboard for:", plot_name)
    path = save_plot_dashboard(plot_name, result, VIS_DIR)
    dashboard_paths.append(path)

print("\nSaved dashboards:")
for p in dashboard_paths:
    print(p)

In [ ]:
def save_index_comparison(index_name, analysis_results, out_dir):
    plot_names = list(analysis_results.keys())

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes = axes.flatten()

    for ax, plot_name in zip(axes, plot_names):
        result = analysis_results[plot_name]

        if index_name == "RGB":
            data = result["rgb_norm"]
            ax.imshow(data, interpolation="nearest")
        else:
            data = result["index_maps"][index_name]
            im = ax.imshow(data, interpolation="nearest")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        ax.set_title(plot_name)
        ax.axis("off")

    plt.suptitle(f"{index_name} comparison across plots", fontsize=16)
    plt.tight_layout()

    save_path = out_dir / f"comparison_{index_name}.png"
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

    return save_path

In [ ]:
comparison_indices = [
    "RGB",
    "NDVI",
    "NDWI",
    "NDMI",
    "NDRE",
    "NBR",
    "GNDVI",
    "MSI",
    "CIRE",
    "GCI",
    "BSI",
    "PSRI",
    "PRI",
    "MTCI",
]

comparison_paths = []

for idx_name in comparison_indices:
    print("Creating comparison for:", idx_name)
    p = save_index_comparison(idx_name, analysis_results, VIS_DIR)
    comparison_paths.append(p)

print("\nSaved comparison figures:")
for p in comparison_paths:
    print(p)

In [ ]:
def save_individual_maps(plot_name, result, out_dir):
    plot_dir = out_dir / plot_name
    plot_dir.mkdir(exist_ok=True)

    # Save RGB
    plt.figure(figsize=(5, 5))
    plt.imshow(result["rgb_norm"], interpolation="nearest")
    plt.title(f"{plot_name} - RGB")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(plot_dir / "RGB.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

    # Save all indices
    for index_name, arr in result["index_maps"].items():
        plt.figure(figsize=(5, 5))
        im = plt.imshow(arr, interpolation="nearest")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        plt.title(f"{plot_name} - {index_name}")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(plot_dir / f"{index_name}.png", dpi=200, bbox_inches="tight")
        plt.show()
        plt.close()

for plot_name, result in analysis_results.items():
    print("Saving individual maps for:", plot_name)
    save_individual_maps(plot_name, result, VIS_DIR)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
def build_pixel_features(
    result,
    selected_indices=None,
    use_full_spectrum=True,
    bad_band_threshold=0.5
):
    """
    Builds a feature matrix for pixel-level segmentation.

    It automatically removes bad spectral bands where too many pixels are NaN.
    """

    cube_hwc = result["cube_hwc"]          # height, width, bands
    index_maps = result["index_maps"]

    h, w, b = cube_hwc.shape

    feature_arrays = []
    feature_names = []

    # Find bad spectral bands
    nan_per_band = np.isnan(cube_hwc).sum(axis=(0, 1))
    total_pixels = h * w
    nan_ratio_per_band = nan_per_band / total_pixels

    bad_band_indices = np.where(nan_ratio_per_band > bad_band_threshold)[0].tolist()
    usable_band_indices = [i for i in range(b) if i not in bad_band_indices]

    print("Bad spectral bands removed:", bad_band_indices)
    print("Usable spectral bands:", len(usable_band_indices))

    if use_full_spectrum:
        for band_idx in usable_band_indices:
            band = cube_hwc[:, :, band_idx]
            feature_arrays.append(band)
            feature_names.append(f"band_{band_idx}")

    if selected_indices is not None:
        for idx_name in selected_indices:
            if idx_name in index_maps:
                feature_arrays.append(index_maps[idx_name])
                feature_names.append(idx_name)
            else:
                print(f"Warning: {idx_name} not found in index_maps")

    # Stack to H x W x F
    features_hwf = np.stack(feature_arrays, axis=-1)

    # Valid pixel = not all features are NaN
    # Πιο ήπιο από το προηγούμενο. Δεν απαιτούμε όλα τα features να είναι non-NaN.
    valid_mask = ~np.isnan(features_hwf).all(axis=-1)

    # Flatten
    X = features_hwf.reshape(-1, features_hwf.shape[-1])
    valid_flat = valid_mask.reshape(-1)

    X_valid = X[valid_flat]

    # Αν μείνουν μεμονωμένα NaN, τα γεμίζουμε με median ανά feature
    X_valid = X_valid.astype(np.float32)

    col_medians = np.nanmedian(X_valid, axis=0)

    # Αν κάποιο feature είναι όλο NaN, βάλε 0
    col_medians = np.nan_to_num(col_medians, nan=0.0)

    inds = np.where(np.isnan(X_valid))
    X_valid[inds] = np.take(col_medians, inds[1])

    return X_valid, valid_mask, feature_names

In [ ]:
def segment_plot_with_pca_kmeans(
    result,
    n_clusters=2,
    n_pca_components=5,
    selected_indices=None,
    use_full_spectrum=True,
    random_state=42
):
    """
    Performs PCA + KMeans segmentation for one plot.

    Returns
    -------
    segmentation_map : np.ndarray
        2D array with cluster labels. NaN for invalid pixels.
    pca_features : np.ndarray
        PCA-transformed features for valid pixels.
    kmeans : fitted KMeans object
    pca : fitted PCA object
    scaler : fitted StandardScaler object
    feature_names : list[str]
    """
    X_valid, valid_mask, feature_names = build_pixel_features(
        result=result,
        selected_indices=selected_indices,
        use_full_spectrum=use_full_spectrum
    )

    print("Valid pixels:", X_valid.shape[0])
    print("Features:", X_valid.shape[1])

    if X_valid.shape[0] < n_clusters:
        raise ValueError("Not enough valid pixels for the requested number of clusters.")

    # Replace inf if any
    X_valid = np.nan_to_num(X_valid, nan=0.0, posinf=0.0, neginf=0.0)

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_valid)

    # PCA
    max_components = min(n_pca_components, X_scaled.shape[0], X_scaled.shape[1])
    pca = PCA(n_components=max_components, random_state=random_state)
    X_pca = pca.fit_transform(X_scaled)

    print("PCA components used:", max_components)
    print("Explained variance ratio:", pca.explained_variance_ratio_)
    print("Total explained variance:", pca.explained_variance_ratio_.sum())

    # KMeans
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = kmeans.fit_predict(X_pca)

    # Reconstruct segmentation map
    h, w = valid_mask.shape
    segmentation_map = np.full((h, w), np.nan)
    segmentation_map.reshape(-1)[valid_mask.reshape(-1)] = labels

    return segmentation_map, X_pca, kmeans, pca, scaler, feature_names

In [ ]:
selected_indices_for_segmentation = [
    "NDVI",
    "NDMI",
    "NDRE",
    "BSI",
    "MSI",
    "CIRE",
    "GCI"
]

plot_name = list(analysis_results.keys())[0]
result = analysis_results[plot_name]

print("Testing segmentation for:", plot_name)

segmentation_map, X_pca, kmeans, pca, scaler, feature_names = segment_plot_with_pca_kmeans(
    result=result,
    n_clusters=2,
    n_pca_components=5,
    selected_indices=selected_indices_for_segmentation,
    use_full_spectrum=True,
    random_state=42
)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(result["rgb_norm"], interpolation="nearest")
plt.title(f"{plot_name} - RGB")
plt.axis("off")
plt.show()

plt.figure(figsize=(6, 6))
im = plt.imshow(segmentation_map, interpolation="nearest")
plt.colorbar(im, label="Cluster")
plt.title(f"{plot_name} - PCA + KMeans segmentation")
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(os.environ.get("MAKEATHON_DATA_DIR", "/content/drive/MyDrive/images_makeathlon"))

csv_files = sorted(list(DATA_DIR.rglob("*.csv")))

print("Number of CSV files found:", len(csv_files))
print()

for i, path in enumerate(csv_files):
    size_kb = path.stat().st_size / 1024
    print(f"{i}: {path.relative_to(DATA_DIR)} | {size_kb:.2f} KB")

In [ ]:
for path in csv_files:
    print("=" * 100)
    print("CSV:", path.relative_to(DATA_DIR))

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:")
        print(list(df.columns))
        print()
        display(df.head())
    except Exception as e:
        print("Error reading file:", e)

In [ ]:
pv_csv_files = [
    p for p in csv_files
    if p.name.upper().startswith("PV") and "plot_spectral" not in p.name
]

print("PV CSV files:", len(pv_csv_files))
for p in pv_csv_files:
    print(p.name)

sample_pv_path = pv_csv_files[0]

print("\nSample file:", sample_pv_path.name)
print("=" * 80)

with open(sample_pv_path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

for i, line in enumerate(lines[:80]):
    print(f"{i:03d}: {line.rstrip()}")

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

def parse_pvgis_csv(path):
    """
    Parses a PVGIS CSV text file and extracts:
    - metadata key-value pairs
    - numeric summary values if present
    - monthly table if present

    Returns:
    - summary_dict
    - tables dict
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw_lines = [line.strip() for line in f.readlines()]

    # Remove fully empty lines but keep structure enough
    lines = [line for line in raw_lines if line.strip() != ""]

    summary = {
        "pv_file": path.name,
    }

    # Infer plot from filename
    name_upper = path.name.upper()
    if "VEROIA" in name_upper:
        summary["plot_name"] = "Veroia-Veroia_20250821_mosaic"
    elif "ARKADIA2" in name_upper:
        summary["plot_name"] = "arkadia2_20240531_mosaic"
    elif "ARKADIA" in name_upper:
        summary["plot_name"] = "arkadia_20241024_mosaic"
    elif "MAGNISIA" in name_upper:
        summary["plot_name"] = "magnisia_20241024_mosaic"
    else:
        summary["plot_name"] = None

    # Extract lat/lon from filename as fallback
    coord_match = re.search(r"_(\d+\.\d+)_(\d+\.\d+)_", path.name)
    if coord_match:
        summary["lat_from_filename"] = float(coord_match.group(1))
        summary["lon_from_filename"] = float(coord_match.group(2))

    # Parse key-value lines like:
    # Location [Lat/Lon]:;37.324;22.144
    for line in lines:
        parts = [p.strip() for p in line.split(";")]

        if len(parts) >= 2:
            key = parts[0].strip()
            values = [p for p in parts[1:] if p != ""]

            if key and values:
                clean_key = (
                    key.replace(":", "")
                       .replace("[", "")
                       .replace("]", "")
                       .replace(" ", "_")
                       .replace("/", "_")
                       .replace("(", "")
                       .replace(")", "")
                       .replace("-", "_")
                       .lower()
                )

                # Store raw joined value
                summary[f"raw_{clean_key}"] = "|".join(values)

                # Try extracting numeric values
                numeric_values = []
                for v in values:
                    try:
                        numeric_values.append(float(v.replace(",", ".")))
                    except:
                        pass

                if len(numeric_values) == 1:
                    summary[clean_key] = numeric_values[0]
                elif len(numeric_values) > 1:
                    for j, num in enumerate(numeric_values):
                        summary[f"{clean_key}_{j+1}"] = num

    # Find monthly/yearly numeric table candidates
    tables = {}

    for i, line in enumerate(lines):
        parts = [p.strip() for p in line.split(";") if p.strip() != ""]

        # common monthly table header contains month or labels like E_m, H(i)_m
        lower = [p.lower() for p in parts]

        if any(x in lower for x in ["month", "monat", "mois"]) or any("e_m" in x.lower() for x in parts):
            header = parts
            data_rows = []

            for j in range(i + 1, len(lines)):
                row_parts = [p.strip() for p in lines[j].split(";") if p.strip() != ""]

                if len(row_parts) < 2:
                    break

                # Stop if a new non-table section starts
                if len(row_parts) == 1 and not row_parts[0].replace(".", "", 1).isdigit():
                    break

                data_rows.append(row_parts)

            if data_rows:
                # normalize row length
                max_len = max(len(r) for r in data_rows)
                header_fixed = header[:max_len]
                if len(header_fixed) < max_len:
                    header_fixed += [f"col_{k}" for k in range(len(header_fixed), max_len)]

                df = pd.DataFrame([r[:max_len] for r in data_rows], columns=header_fixed)

                # convert numeric columns where possible
                for col in df.columns:
                    df[col] = pd.to_numeric(
                        df[col].astype(str).str.replace(",", ".", regex=False),
                        errors="ignore"
                    )

                tables[f"table_at_line_{i}"] = df

    return summary, tables

In [ ]:
pv_summaries = []
pv_tables_by_file = {}

for path in pv_csv_files:
    summary, tables = parse_pvgis_csv(path)
    pv_summaries.append(summary)
    pv_tables_by_file[path.name] = tables

pv_summary_df = pd.DataFrame(pv_summaries)

print("PV summary shape:", pv_summary_df.shape)
display(pv_summary_df.head())
print("Columns:")
print(list(pv_summary_df.columns))

In [ ]:
for col in pv_summary_df.columns:
    print(col)

In [ ]:
for file_name, tables in pv_tables_by_file.items():
    print("=" * 100)
    print(file_name)
    print("Number of tables found:", len(tables))

    for table_name, df in tables.items():
        print("\nTable:", table_name)
        print("Shape:", df.shape)
        display(df.head())

In [ ]:
indices_summary_v2_df = pd.read_csv(DATA_DIR / "outputs" / "plot_spectral_indices_extended.csv")

merged_df = indices_summary_v2_df.merge(
    pv_summary_df,
    on="plot_name",
    how="left"
)

display(merged_df[
    [
        "plot_name",
        "NDVI_mean",
        "NDMI_mean",
        "BSI_mean",
        "MSI_mean",
        "CIRE_mean",
        "yearly_pv_energy_production_kwh",
        "yearly_in_plane_irradiation_kwh_m2",
        "total_loss_%",
    ]
])

In [ ]:
def minmax_score(series, higher_is_better=True):
    """
    Converts a numeric pandas Series to 0-1 using min-max scaling.
    """
    s = pd.to_numeric(series, errors="coerce")

    min_val = s.min()
    max_val = s.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series([0.5] * len(s), index=s.index)

    score = (s - min_val) / (max_val - min_val)

    if not higher_is_better:
        score = 1 - score

    return score

In [ ]:
scored_df = merged_df.copy()

# Vegetation / agriculture components
scored_df["score_ndvi_agri"] = minmax_score(scored_df["NDVI_mean"], higher_is_better=True)
scored_df["score_ndmi_agri"] = minmax_score(scored_df["NDMI_mean"], higher_is_better=True)
scored_df["score_cire_agri"] = minmax_score(scored_df["CIRE_mean"], higher_is_better=True)
scored_df["score_gci_agri"] = minmax_score(scored_df["GCI_mean"], higher_is_better=True)
scored_df["score_msi_agri"] = minmax_score(scored_df["MSI_mean"], higher_is_better=False)
scored_df["score_bsi_agri"] = minmax_score(scored_df["BSI_mean"], higher_is_better=False)

scored_df["agriculture_score"] = 100 * (
    0.35 * scored_df["score_ndvi_agri"] +
    0.20 * scored_df["score_ndmi_agri"] +
    0.15 * scored_df["score_cire_agri"] +
    0.10 * scored_df["score_gci_agri"] +
    0.10 * scored_df["score_msi_agri"] +
    0.10 * scored_df["score_bsi_agri"]
)

In [ ]:
# PVGIS components
scored_df["score_pv_production"] = minmax_score(
    scored_df["yearly_pv_energy_production_kwh"],
    higher_is_better=True
)

scored_df["score_irradiation"] = minmax_score(
    scored_df["yearly_in_plane_irradiation_kwh_m2"],
    higher_is_better=True
)

scored_df["score_total_loss"] = minmax_score(
    scored_df["total_loss_%"],
    higher_is_better=False
)

# Spectral components for PV
scored_df["score_low_ndvi_pv"] = minmax_score(scored_df["NDVI_mean"], higher_is_better=False)
scored_df["score_low_ndmi_pv"] = minmax_score(scored_df["NDMI_mean"], higher_is_better=False)
scored_df["score_high_bsi_pv"] = minmax_score(scored_df["BSI_mean"], higher_is_better=True)
scored_df["score_high_msi_pv"] = minmax_score(scored_df["MSI_mean"], higher_is_better=True)

scored_df["pv_score"] = 100 * (
    0.30 * scored_df["score_pv_production"] +
    0.20 * scored_df["score_irradiation"] +
    0.15 * scored_df["score_total_loss"] +
    0.15 * scored_df["score_low_ndvi_pv"] +
    0.08 * scored_df["score_low_ndmi_pv"] +
    0.07 * scored_df["score_high_bsi_pv"] +
    0.05 * scored_df["score_high_msi_pv"]
)

In [ ]:
def classify_recommendation(row):
    agri = row["agriculture_score"]
    pv = row["pv_score"]

    diff = abs(agri - pv)

    if agri >= 70 and agri > pv:
        return "agriculture_priority"

    if pv >= 70 and pv > agri:
        return "pv_priority"

    if diff < 15:
        return "mixed_use_or_needs_more_context"

    if agri > pv:
        return "agriculture_candidate"

    return "pv_candidate"


def confidence_level(row):
    diff = abs(row["agriculture_score"] - row["pv_score"])

    if diff >= 35:
        return "high"
    elif diff >= 20:
        return "medium"
    else:
        return "low"


scored_df["suggested_use"] = scored_df.apply(classify_recommendation, axis=1)
scored_df["recommendation_confidence"] = scored_df.apply(confidence_level, axis=1)

In [ ]:
final_cols = [
    "plot_name",

    "NDVI_mean",
    "NDMI_mean",
    "BSI_mean",
    "MSI_mean",
    "CIRE_mean",

    "yearly_pv_energy_production_kwh",
    "yearly_in_plane_irradiation_kwh_m2",
    "total_loss_%",

    "agriculture_score",
    "pv_score",
    "suggested_use",
    "recommendation_confidence",
]

final_recommendation_df = scored_df[final_cols].copy()

display(
    final_recommendation_df.sort_values(
        by=["suggested_use", "pv_score"],
        ascending=[True, False]
    )
)

In [ ]:
final_path = OUTPUT_DIR / "final_land_use_recommendations.csv"

final_recommendation_df.to_csv(final_path, index=False)

print("Saved final recommendations to:")
print(final_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_labels = final_recommendation_df["plot_name"].tolist()
x = np.arange(len(plot_labels))
width = 0.35

plt.figure(figsize=(12, 6))

plt.bar(x - width/2, final_recommendation_df["agriculture_score"], width, label="Agriculture score")
plt.bar(x + width/2, final_recommendation_df["pv_score"], width, label="PV score")

plt.xticks(x, plot_labels, rotation=30, ha="right")
plt.ylabel("Score")
plt.title("Agriculture vs PV suitability score per plot")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()

score_plot_path = OUTPUT_DIR / "agriculture_vs_pv_scores.png"
plt.savefig(score_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(score_plot_path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(os.environ.get("MAKEATHON_DATA_DIR", "/content/drive/MyDrive/images_makeathlon"))

xlsx_files = sorted(list(DATA_DIR.rglob("*.xlsx")) + list(DATA_DIR.rglob("*.xls")))

print("Excel files found:", len(xlsx_files))
for i, path in enumerate(xlsx_files):
    print(f"{i}: {path.relative_to(DATA_DIR)}")

In [ ]:
wind_path = DATA_DIR / "wind_power_densities.xlsx"

print("Exists:", wind_path.exists())
print("Path:", wind_path)

xls = pd.ExcelFile(wind_path)

print("Sheet names:")
print(xls.sheet_names)

In [ ]:
for sheet in xls.sheet_names:
    print("=" * 100)
    print("Sheet:", sheet)

    df = pd.read_excel(wind_path, sheet_name=sheet)

    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))

    display(df.head(10))

In [ ]:
wind_df = pd.read_excel(wind_path, sheet_name="Φύλλο1")

wind_df = wind_df.rename(columns={
    "Name": "wind_name",
    "LAT": "raw_lon_probably",
    "LON": "raw_lat_probably",
    "MEAN POWER DENSITY (W/m^2)": "wind_power_density_w_m2"
})

# Διόρθωση γιατί οι στήλες φαίνονται swapped
wind_df["wind_lat"] = wind_df["raw_lat_probably"]
wind_df["wind_lon"] = wind_df["raw_lon_probably"]

wind_df

In [ ]:
wind_name_to_plot_name = {
    "Arkadia": "arkadia_20241024_mosaic",
    "Magnisia": "magnisia_20241024_mosaic",
    "Arkadia 2": "arkadia2_20240531_mosaic",
    "Veroia": "Veroia-Veroia_20250821_mosaic",
}

wind_df["plot_name"] = wind_df["wind_name"].map(wind_name_to_plot_name)

wind_df[
    [
        "plot_name",
        "wind_name",
        "wind_lat",
        "wind_lon",
        "wind_power_density_w_m2"
    ]
]

In [ ]:
wind_df["wind_resource_score"] = 100 * minmax_score(
    wind_df["wind_power_density_w_m2"],
    higher_is_better=True
)

wind_df[
    [
        "plot_name",
        "wind_power_density_w_m2",
        "wind_resource_score"
    ]
].sort_values("wind_resource_score", ascending=False)

In [ ]:
final_multi_df = scored_df.merge(
    wind_df[
        [
            "plot_name",
            "wind_name",
            "wind_lat",
            "wind_lon",
            "wind_power_density_w_m2",
            "wind_resource_score"
        ]
    ],
    on="plot_name",
    how="left"
)

display(final_multi_df[
    [
        "plot_name",
        "agriculture_score",
        "pv_score",
        "wind_power_density_w_m2",
        "wind_resource_score"
    ]
])

In [ ]:
final_multi_df["score_low_ndvi_wind"] = minmax_score(
    final_multi_df["NDVI_mean"],
    higher_is_better=False
)

final_multi_df["score_low_agri_wind"] = minmax_score(
    final_multi_df["agriculture_score"],
    higher_is_better=False
)

final_multi_df["wind_score"] = (
    0.70 * final_multi_df["wind_resource_score"] +
    0.20 * 100 * final_multi_df["score_low_ndvi_wind"] +
    0.10 * 100 * final_multi_df["score_low_agri_wind"]
)

In [ ]:
def final_best_use(row):
    scores = {
        "agriculture_priority": row["agriculture_score"],
        "pv_priority": row["pv_score"],
        "wind_priority": row["wind_score"],
    }

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    sorted_scores = sorted(scores.values(), reverse=True)
    diff = sorted_scores[0] - sorted_scores[1]

    # Αν PV και Wind είναι και τα δύο δυνατά, δώσε hybrid recommendation
    if row["pv_score"] >= 65 and row["wind_score"] >= 65:
        return "renewable_energy_hybrid_candidate"

    if diff < 10:
        return "mixed_use_or_needs_more_context"

    return best_label


def final_confidence(row):
    scores = [
        row["agriculture_score"],
        row["pv_score"],
        row["wind_score"]
    ]

    sorted_scores = sorted(scores, reverse=True)
    diff = sorted_scores[0] - sorted_scores[1]

    if diff >= 30:
        return "high"
    elif diff >= 15:
        return "medium"
    else:
        return "low"


final_multi_df["final_recommendation"] = final_multi_df.apply(final_best_use, axis=1)
final_multi_df["final_confidence"] = final_multi_df.apply(final_confidence, axis=1)

In [ ]:
final_cols = [
    "plot_name",

    "NDVI_mean",
    "NDMI_mean",
    "BSI_mean",

    "yearly_pv_energy_production_kwh",
    "yearly_in_plane_irradiation_kwh_m2",
    "total_loss_%",

    "wind_power_density_w_m2",

    "agriculture_score",
    "pv_score",
    "wind_score",

    "final_recommendation",
    "final_confidence",
]

final_decision_df = final_multi_df[final_cols].copy()

display(
    final_decision_df.sort_values(
        by=["final_recommendation", "wind_score"],
        ascending=[True, False]
    )
)

In [ ]:
final_decision_path = OUTPUT_DIR / "final_agriculture_pv_wind_recommendations.csv"

final_decision_df.to_csv(final_decision_path, index=False)

print("Saved final decision CSV to:")
print(final_decision_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_labels = final_decision_df["plot_name"].tolist()
x = np.arange(len(plot_labels))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, final_decision_df["agriculture_score"], width, label="Agriculture")
plt.bar(x, final_decision_df["pv_score"], width, label="PV")
plt.bar(x + width, final_decision_df["wind_score"], width, label="Wind")

plt.xticks(x, plot_labels, rotation=30, ha="right")
plt.ylabel("Score")
plt.title("Agriculture vs PV vs Wind suitability per plot")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()

multi_score_plot_path = OUTPUT_DIR / "agriculture_pv_wind_scores.png"
plt.savefig(multi_score_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(multi_score_plot_path)

In [ ]:
def wind_power_density_absolute_score(wpd):
    """
    Absolute wind suitability score from 0 to 100 based on WPD in W/m².

    This is not relative among candidate plots.
    It reflects whether the wind resource is actually strong enough.
    """
    if pd.isna(wpd):
        return np.nan

    if wpd < 100:
        return 5
    elif wpd < 200:
        # 100 -> 10, 200 -> 25
        return 10 + (wpd - 100) / 100 * 15
    elif wpd < 300:
        # 200 -> 25, 300 -> 45
        return 25 + (wpd - 200) / 100 * 20
    elif wpd < 400:
        # 300 -> 45, 400 -> 65
        return 45 + (wpd - 300) / 100 * 20
    elif wpd < 600:
        # 400 -> 65, 600 -> 85
        return 65 + (wpd - 400) / 200 * 20
    else:
        # 600+ -> 85-100, capped
        return min(100, 85 + (wpd - 600) / 400 * 15)

In [ ]:
final_multi_df["wind_resource_absolute_score"] = final_multi_df[
    "wind_power_density_w_m2"
].apply(wind_power_density_absolute_score)

final_multi_df[
    ["plot_name", "wind_power_density_w_m2", "wind_resource_absolute_score"]
].sort_values("wind_resource_absolute_score", ascending=False)

In [ ]:
final_multi_df["score_low_ndvi_wind"] = 100 * minmax_score(
    final_multi_df["NDVI_mean"],
    higher_is_better=False
)

final_multi_df["score_low_agri_wind"] = 100 * minmax_score(
    final_multi_df["agriculture_score"],
    higher_is_better=False
)

final_multi_df["wind_score"] = (
    0.80 * final_multi_df["wind_resource_absolute_score"] +
    0.10 * final_multi_df["score_low_ndvi_wind"] +
    0.10 * final_multi_df["score_low_agri_wind"]
)

In [ ]:
def apply_wind_caps(row):
    """
    Prevents low-wind plots from getting high wind suitability
    just because vegetation/agriculture score is low.
    """
    wpd = row["wind_power_density_w_m2"]
    score = row["wind_score"]

    if pd.isna(wpd):
        return np.nan

    if wpd < 200:
        return min(score, 30)
    elif wpd < 300:
        return min(score, 45)
    elif wpd < 400:
        return min(score, 65)
    else:
        return score

final_multi_df["wind_score"] = final_multi_df.apply(apply_wind_caps, axis=1)

In [ ]:
def wind_resource_class(wpd):
    if pd.isna(wpd):
        return "unknown"
    elif wpd < 200:
        return "poor_wind_resource"
    elif wpd < 300:
        return "marginal_wind_resource"
    elif wpd < 400:
        return "moderate_wind_resource"
    elif wpd < 600:
        return "good_wind_resource"
    else:
        return "very_good_wind_resource"

final_multi_df["wind_resource_class"] = final_multi_df[
    "wind_power_density_w_m2"
].apply(wind_resource_class)

In [ ]:
def pv_production_absolute_score(kwh_per_kwp_year):
    """
    Absolute PV suitability score based on yearly PV production
    for 1 kWp installed capacity.
    """
    if pd.isna(kwh_per_kwp_year):
        return np.nan

    if kwh_per_kwp_year < 1200:
        return 20
    elif kwh_per_kwp_year < 1400:
        return 20 + (kwh_per_kwp_year - 1200) / 200 * 25
    elif kwh_per_kwp_year < 1600:
        return 45 + (kwh_per_kwp_year - 1400) / 200 * 25
    elif kwh_per_kwp_year < 1800:
        return 70 + (kwh_per_kwp_year - 1600) / 200 * 20
    else:
        return min(100, 90 + (kwh_per_kwp_year - 1800) / 300 * 10)


def pv_loss_absolute_score(total_loss_pct):
    """
    Lower losses are better.
    """
    if pd.isna(total_loss_pct):
        return np.nan

    if total_loss_pct <= 14:
        return 100
    elif total_loss_pct <= 18:
        return 85
    elif total_loss_pct <= 22:
        return 65
    elif total_loss_pct <= 26:
        return 45
    else:
        return 25

In [ ]:
final_multi_df["pv_production_absolute_score"] = final_multi_df[
    "yearly_pv_energy_production_kwh"
].apply(pv_production_absolute_score)

final_multi_df["pv_loss_absolute_score"] = final_multi_df[
    "total_loss_%"
].apply(pv_loss_absolute_score)

In [ ]:
final_multi_df["score_low_ndvi_pv"] = 100 * minmax_score(
    final_multi_df["NDVI_mean"],
    higher_is_better=False
)

final_multi_df["score_low_ndmi_pv"] = 100 * minmax_score(
    final_multi_df["NDMI_mean"],
    higher_is_better=False
)

final_multi_df["score_high_bsi_pv"] = 100 * minmax_score(
    final_multi_df["BSI_mean"],
    higher_is_better=True
)

final_multi_df["pv_score"] = (
    0.60 * final_multi_df["pv_production_absolute_score"] +
    0.20 * final_multi_df["pv_loss_absolute_score"] +
    0.10 * final_multi_df["score_low_ndvi_pv"] +
    0.05 * final_multi_df["score_low_ndmi_pv"] +
    0.05 * final_multi_df["score_high_bsi_pv"]
)

In [ ]:
def ndvi_agri_absolute_score(ndvi):
    if pd.isna(ndvi):
        return np.nan

    if ndvi < 0.2:
        return 10
    elif ndvi < 0.4:
        return 30 + (ndvi - 0.2) / 0.2 * 20
    elif ndvi < 0.6:
        return 50 + (ndvi - 0.4) / 0.2 * 20
    elif ndvi < 0.75:
        return 70 + (ndvi - 0.6) / 0.15 * 15
    else:
        return min(100, 85 + (ndvi - 0.75) / 0.25 * 15)


def ndmi_agri_absolute_score(ndmi):
    if pd.isna(ndmi):
        return np.nan

    if ndmi < -0.1:
        return 20
    elif ndmi < 0.1:
        return 40 + (ndmi + 0.1) / 0.2 * 20
    elif ndmi < 0.25:
        return 60 + (ndmi - 0.1) / 0.15 * 20
    else:
        return min(100, 80 + (ndmi - 0.25) / 0.25 * 20)

In [ ]:
final_multi_df["ndvi_agri_absolute_score"] = final_multi_df["NDVI_mean"].apply(
    ndvi_agri_absolute_score
)

final_multi_df["ndmi_agri_absolute_score"] = final_multi_df["NDMI_mean"].apply(
    ndmi_agri_absolute_score
)

# CIRE/GCI ακόμα τα κρατάμε relative γιατί thresholds εξαρτώνται περισσότερο από sensor/data scaling
final_multi_df["score_cire_agri"] = 100 * minmax_score(
    final_multi_df["CIRE_mean"],
    higher_is_better=True
)

final_multi_df["score_gci_agri"] = 100 * minmax_score(
    final_multi_df["GCI_mean"],
    higher_is_better=True
)

final_multi_df["agriculture_score"] = (
    0.45 * final_multi_df["ndvi_agri_absolute_score"] +
    0.30 * final_multi_df["ndmi_agri_absolute_score"] +
    0.15 * final_multi_df["score_cire_agri"] +
    0.10 * final_multi_df["score_gci_agri"]
)

In [ ]:
def final_best_use_absolute(row):
    agri = row["agriculture_score"]
    pv = row["pv_score"]
    wind = row["wind_score"]

    # Hard rule: αν wind resource είναι poor, μην προτείνεις wind
    if row["wind_power_density_w_m2"] < 200:
        wind = 0

    scores = {
        "agriculture_priority": agri,
        "pv_priority": pv,
        "wind_priority": wind,
    }

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    sorted_scores = sorted(scores.values(), reverse=True)
    diff = sorted_scores[0] - sorted_scores[1]

    if best_score < 50:
        return "no_strong_candidate"

    if diff < 10:
        return "mixed_use_or_needs_more_context"

    return best_label


final_multi_df["final_recommendation"] = final_multi_df.apply(
    final_best_use_absolute,
    axis=1
)

final_multi_df["final_confidence"] = final_multi_df.apply(
    final_confidence,
    axis=1
)

In [ ]:
final_cols = [
    "plot_name",

    "NDVI_mean",
    "NDMI_mean",
    "BSI_mean",

    "yearly_pv_energy_production_kwh",
    "total_loss_%",

    "wind_power_density_w_m2",
    "wind_resource_class",

    "agriculture_score",
    "pv_score",
    "wind_score",

    "final_recommendation",
    "final_confidence",
]

final_decision_df = final_multi_df[final_cols].copy()

display(final_decision_df)

In [ ]:
final_decision_path = OUTPUT_DIR / "final_absolute_agriculture_pv_wind_recommendations.csv"

final_decision_df.to_csv(final_decision_path, index=False)

print("Saved final decision CSV to:")
print(final_decision_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(os.environ.get("MAKEATHON_DATA_DIR", "/content/drive/MyDrive/images_makeathlon"))
OUTPUT_DIR = DATA_DIR / "outputs"

final_decision_path = OUTPUT_DIR / "final_absolute_agriculture_pv_wind_recommendations.csv"

final_decision_df = pd.read_csv(final_decision_path)

display(final_decision_df)

In [ ]:
plot_labels = final_decision_df["plot_name"].tolist()
x = np.arange(len(plot_labels))
width = 0.25

plt.figure(figsize=(13, 6))

plt.bar(x - width, final_decision_df["agriculture_score"], width, label="Agriculture")
plt.bar(x, final_decision_df["pv_score"], width, label="PV")
plt.bar(x + width, final_decision_df["wind_score"], width, label="Wind")

plt.xticks(x, plot_labels, rotation=30, ha="right")
plt.ylabel("Suitability score")
plt.title("Agriculture vs PV vs Wind suitability per plot")
plt.ylim(0, 100)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

score_plot_path = OUTPUT_DIR / "absolute_agriculture_pv_wind_scores.png"
plt.savefig(score_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(score_plot_path)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

score_columns = [
    ("agriculture_score", "Agriculture ranking"),
    ("pv_score", "PV ranking"),
    ("wind_score", "Wind ranking"),
]

for ax, (col, title) in zip(axes, score_columns):
    sorted_df = final_decision_df.sort_values(col, ascending=True)

    ax.barh(sorted_df["plot_name"], sorted_df[col])
    ax.set_xlim(0, 100)
    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()

ranking_plot_path = OUTPUT_DIR / "absolute_rankings_by_use.png"
plt.savefig(ranking_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(ranking_plot_path)

In [ ]:
wind_sorted_df = final_decision_df.sort_values(
    "wind_power_density_w_m2",
    ascending=False
)

plt.figure(figsize=(10, 5))

plt.bar(
    wind_sorted_df["plot_name"],
    wind_sorted_df["wind_power_density_w_m2"]
)

plt.axhline(200, linestyle="--", label="Poor / marginal threshold: 200 W/m²")
plt.axhline(300, linestyle="--", label="Marginal / moderate threshold: 300 W/m²")
plt.axhline(400, linestyle="--", label="Moderate / good threshold: 400 W/m²")

plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean Power Density (W/m²)")
plt.title("Wind power density per plot")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

wind_plot_path = OUTPUT_DIR / "wind_power_density_thresholds.png"
plt.savefig(wind_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(wind_plot_path)

In [ ]:
pv_sorted_df = final_decision_df.sort_values(
    "yearly_pv_energy_production_kwh",
    ascending=False
)

plt.figure(figsize=(10, 5))

plt.bar(
    pv_sorted_df["plot_name"],
    pv_sorted_df["yearly_pv_energy_production_kwh"]
)

plt.axhline(1200, linestyle="--", label="Weak: 1200 kWh/kWp/year")
plt.axhline(1400, linestyle="--", label="Moderate: 1400")
plt.axhline(1600, linestyle="--", label="Good: 1600")
plt.axhline(1800, linestyle="--", label="Excellent: 1800")

plt.xticks(rotation=30, ha="right")
plt.ylabel("Yearly PV energy production (kWh/kWp/year)")
plt.title("PVGIS yearly PV production per plot")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

pv_plot_path = OUTPUT_DIR / "pv_yearly_production_thresholds.png"
plt.savefig(pv_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(pv_plot_path)

In [ ]:
x = np.arange(len(final_decision_df))
width = 0.35

plt.figure(figsize=(12, 5))

plt.bar(x - width/2, final_decision_df["NDVI_mean"], width, label="NDVI mean")
plt.bar(x + width/2, final_decision_df["NDMI_mean"], width, label="NDMI mean")

plt.axhline(0.2, linestyle="--", label="Low vegetation threshold")
plt.axhline(0.5, linestyle="--", label="Healthy vegetation threshold")

plt.xticks(x, final_decision_df["plot_name"], rotation=30, ha="right")
plt.ylabel("Index value")
plt.title("Vegetation and moisture indicators per plot")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

veg_plot_path = OUTPUT_DIR / "vegetation_moisture_indices.png"
plt.savefig(veg_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(veg_plot_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ.get("MAKEATHON_DATA_DIR", "/content/drive/MyDrive/images_makeathlon"))
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

infrastructure_data = [
    {
        "plot_name": "Veroia-Veroia_20250821_mosaic",
        "distance_to_substation_km": 4.2,
        "nearest_voltage_kv": 150,
        "available_grid_capacity_mw": 18,
        "distance_to_transmission_line_km": 2.1,
        "distance_to_road_km": 0.6,
        "distance_to_fiber_km": 7.5,
        "distance_to_city_km": 6.0,
        "flood_risk_score": 25,
        "wildfire_risk_score": 35,
        "seismic_risk_score": 40,
        "protected_area_flag": 0,
        "nearby_res_capacity_mw": 35,
        "curtailment_risk_score": 45,
        "price_spread_score": 60,
        "water_availability_score": 55,
        "power_reliability_score": 70,
        "cooling_climate_score": 55,
    },
    {
        "plot_name": "arkadia2_20240531_mosaic",
        "distance_to_substation_km": 7.8,
        "nearest_voltage_kv": 150,
        "available_grid_capacity_mw": 12,
        "distance_to_transmission_line_km": 4.5,
        "distance_to_road_km": 1.2,
        "distance_to_fiber_km": 12.0,
        "distance_to_city_km": 14.0,
        "flood_risk_score": 20,
        "wildfire_risk_score": 55,
        "seismic_risk_score": 45,
        "protected_area_flag": 0,
        "nearby_res_capacity_mw": 50,
        "curtailment_risk_score": 65,
        "price_spread_score": 70,
        "water_availability_score": 40,
        "power_reliability_score": 60,
        "cooling_climate_score": 50,
    },
    {
        "plot_name": "arkadia_20241024_mosaic",
        "distance_to_substation_km": 3.0,
        "nearest_voltage_kv": 150,
        "available_grid_capacity_mw": 25,
        "distance_to_transmission_line_km": 1.4,
        "distance_to_road_km": 0.4,
        "distance_to_fiber_km": 9.0,
        "distance_to_city_km": 9.5,
        "flood_risk_score": 30,
        "wildfire_risk_score": 45,
        "seismic_risk_score": 45,
        "protected_area_flag": 0,
        "nearby_res_capacity_mw": 60,
        "curtailment_risk_score": 75,
        "price_spread_score": 80,
        "water_availability_score": 45,
        "power_reliability_score": 75,
        "cooling_climate_score": 50,
    },
    {
        "plot_name": "magnisia_20241024_mosaic",
        "distance_to_substation_km": 5.5,
        "nearest_voltage_kv": 150,
        "available_grid_capacity_mw": 20,
        "distance_to_transmission_line_km": 3.0,
        "distance_to_road_km": 0.8,
        "distance_to_fiber_km": 5.0,
        "distance_to_city_km": 7.0,
        "flood_risk_score": 45,
        "wildfire_risk_score": 40,
        "seismic_risk_score": 50,
        "protected_area_flag": 0,
        "nearby_res_capacity_mw": 45,
        "curtailment_risk_score": 55,
        "price_spread_score": 65,
        "water_availability_score": 65,
        "power_reliability_score": 72,
        "cooling_climate_score": 60,
    },
]

infrastructure_df = pd.DataFrame(infrastructure_data)

infra_path = DATA_DIR / "infrastructure_synthetic_indicators.csv"
infrastructure_df.to_csv(infra_path, index=False)

print("Saved synthetic infrastructure CSV to:")
print(infra_path)

display(infrastructure_df)

In [ ]:
final_decision_path = OUTPUT_DIR / "final_absolute_agriculture_pv_wind_recommendations.csv"

final_decision_df = pd.read_csv(final_decision_path)

display(final_decision_df)

In [ ]:
infrastructure_df = pd.read_csv(DATA_DIR / "infrastructure_synthetic_indicators.csv")

multiuse_df = final_decision_df.merge(
    infrastructure_df,
    on="plot_name",
    how="left"
)

display(multiuse_df.head())

In [ ]:
def lower_distance_score(distance_km, excellent=1, good=5, acceptable=15, max_distance=30):
    """
    Lower distance is better.
    Returns 0-100.
    """
    if pd.isna(distance_km):
        return np.nan

    if distance_km <= excellent:
        return 100
    elif distance_km <= good:
        return 80 + (good - distance_km) / (good - excellent) * 20
    elif distance_km <= acceptable:
        return 50 + (acceptable - distance_km) / (acceptable - good) * 30
    elif distance_km <= max_distance:
        return 20 + (max_distance - distance_km) / (max_distance - acceptable) * 30
    else:
        return 10


def higher_capacity_score(value, weak=5, moderate=15, good=30, excellent=60):
    """
    Higher capacity is better.
    Returns 0-100.
    """
    if pd.isna(value):
        return np.nan

    if value < weak:
        return 10
    elif value < moderate:
        return 20 + (value - weak) / (moderate - weak) * 30
    elif value < good:
        return 50 + (value - moderate) / (good - moderate) * 30
    elif value < excellent:
        return 80 + (value - good) / (excellent - good) * 20
    else:
        return 100


def inverse_risk_score(risk_score):
    """
    Input risk score: 0 low risk, 100 high risk.
    Output suitability: 100 low risk, 0 high risk.
    """
    if pd.isna(risk_score):
        return np.nan

    return float(np.clip(100 - risk_score, 0, 100))


def direct_score(value):
    """
    For indicators already given as 0-100, higher is better.
    """
    if pd.isna(value):
        return np.nan

    return float(np.clip(value, 0, 100))

In [ ]:
multiuse_df["bess_substation_score"] = multiuse_df["distance_to_substation_km"].apply(
    lambda x: lower_distance_score(x, excellent=1, good=5, acceptable=12, max_distance=25)
)

multiuse_df["bess_transmission_score"] = multiuse_df["distance_to_transmission_line_km"].apply(
    lambda x: lower_distance_score(x, excellent=1, good=4, acceptable=10, max_distance=20)
)

multiuse_df["bess_grid_capacity_score"] = multiuse_df["available_grid_capacity_mw"].apply(
    lambda x: higher_capacity_score(x, weak=5, moderate=15, good=30, excellent=60)
)

multiuse_df["bess_price_spread_score"] = multiuse_df["price_spread_score"].apply(direct_score)

multiuse_df["bess_curtailment_score"] = multiuse_df["curtailment_risk_score"].apply(direct_score)

multiuse_df["bess_nearby_res_score"] = multiuse_df["nearby_res_capacity_mw"].apply(
    lambda x: higher_capacity_score(x, weak=10, moderate=30, good=80, excellent=150)
)

multiuse_df["bess_road_access_score"] = multiuse_df["distance_to_road_km"].apply(
    lambda x: lower_distance_score(x, excellent=0.5, good=2, acceptable=6, max_distance=15)
)

multiuse_df["bess_flood_safety_score"] = multiuse_df["flood_risk_score"].apply(inverse_risk_score)
multiuse_df["bess_fire_safety_score"] = multiuse_df["wildfire_risk_score"].apply(inverse_risk_score)

In [ ]:
multiuse_df["battery_score_raw"] = (
    0.22 * multiuse_df["bess_substation_score"] +
    0.15 * multiuse_df["bess_transmission_score"] +
    0.18 * multiuse_df["bess_grid_capacity_score"] +
    0.15 * multiuse_df["bess_price_spread_score"] +
    0.10 * multiuse_df["bess_curtailment_score"] +
    0.08 * multiuse_df["bess_nearby_res_score"] +
    0.07 * multiuse_df["bess_road_access_score"] +
    0.03 * multiuse_df["bess_flood_safety_score"] +
    0.02 * multiuse_df["bess_fire_safety_score"]
)

In [ ]:
def apply_battery_caps(row):
    score = row["battery_score_raw"]

    # Protected area = not suitable
    if row["protected_area_flag"] == 1:
        return min(score, 20)

    # If too far from substation, cap
    if row["distance_to_substation_km"] > 15:
        score = min(score, 45)

    # If grid capacity too low, cap
    if row["available_grid_capacity_mw"] < 5:
        score = min(score, 35)

    # If very high flood or fire risk, cap
    if row["flood_risk_score"] > 75 or row["wildfire_risk_score"] > 75:
        score = min(score, 40)

    return score

multiuse_df["battery_score"] = multiuse_df.apply(apply_battery_caps, axis=1)

In [ ]:
multiuse_df["dc_power_capacity_score"] = multiuse_df["available_grid_capacity_mw"].apply(
    lambda x: higher_capacity_score(x, weak=10, moderate=30, good=80, excellent=150)
)

multiuse_df["dc_substation_score"] = multiuse_df["distance_to_substation_km"].apply(
    lambda x: lower_distance_score(x, excellent=1, good=5, acceptable=15, max_distance=30)
)

multiuse_df["dc_transmission_score"] = multiuse_df["distance_to_transmission_line_km"].apply(
    lambda x: lower_distance_score(x, excellent=1, good=5, acceptable=15, max_distance=30)
)

multiuse_df["dc_fiber_score"] = multiuse_df["distance_to_fiber_km"].apply(
    lambda x: lower_distance_score(x, excellent=1, good=5, acceptable=15, max_distance=35)
)

multiuse_df["dc_road_access_score"] = multiuse_df["distance_to_road_km"].apply(
    lambda x: lower_distance_score(x, excellent=0.5, good=2, acceptable=6, max_distance=15)
)

multiuse_df["dc_city_access_score"] = multiuse_df["distance_to_city_km"].apply(
    lambda x: lower_distance_score(x, excellent=5, good=15, acceptable=40, max_distance=80)
)

multiuse_df["dc_flood_safety_score"] = multiuse_df["flood_risk_score"].apply(inverse_risk_score)
multiuse_df["dc_fire_safety_score"] = multiuse_df["wildfire_risk_score"].apply(inverse_risk_score)
multiuse_df["dc_seismic_safety_score"] = multiuse_df["seismic_risk_score"].apply(inverse_risk_score)

multiuse_df["dc_water_score"] = multiuse_df["water_availability_score"].apply(direct_score)
multiuse_df["dc_power_reliability_score"] = multiuse_df["power_reliability_score"].apply(direct_score)
multiuse_df["dc_cooling_score"] = multiuse_df["cooling_climate_score"].apply(direct_score)

In [ ]:
multiuse_df["data_center_score_raw"] = (
    0.22 * multiuse_df["dc_power_capacity_score"] +
    0.12 * multiuse_df["dc_power_reliability_score"] +
    0.12 * multiuse_df["dc_substation_score"] +
    0.08 * multiuse_df["dc_transmission_score"] +
    0.16 * multiuse_df["dc_fiber_score"] +
    0.06 * multiuse_df["dc_road_access_score"] +
    0.05 * multiuse_df["dc_city_access_score"] +
    0.05 * multiuse_df["dc_water_score"] +
    0.05 * multiuse_df["dc_cooling_score"] +
    0.04 * multiuse_df["dc_flood_safety_score"] +
    0.03 * multiuse_df["dc_fire_safety_score"] +
    0.02 * multiuse_df["dc_seismic_safety_score"]
)

In [ ]:
def apply_data_center_caps(row):
    score = row["data_center_score_raw"]

    if row["protected_area_flag"] == 1:
        return min(score, 15)

    # Data centers are power-hungry; low capacity should cap strongly
    if row["available_grid_capacity_mw"] < 10:
        score = min(score, 35)

    # Too far from fiber is a big issue
    if row["distance_to_fiber_km"] > 20:
        score = min(score, 45)

    # Very high flood/fire/seismic risk should cap
    if row["flood_risk_score"] > 75:
        score = min(score, 40)

    if row["wildfire_risk_score"] > 80:
        score = min(score, 45)

    if row["seismic_risk_score"] > 80:
        score = min(score, 45)

    return score

multiuse_df["data_center_score"] = multiuse_df.apply(apply_data_center_caps, axis=1)

In [ ]:
def final_best_use_5(row):
    scores = {
        "agriculture_priority": row["agriculture_score"],
        "pv_priority": row["pv_score"],
        "wind_priority": row["wind_score"],
        "battery_storage_candidate": row["battery_score"],
        "data_center_candidate": row["data_center_score"],
    }

    # Hard rule: if wind resource is poor, don't let wind win
    if row["wind_power_density_w_m2"] < 200:
        scores["wind_priority"] = 0

    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]

    sorted_values = sorted(scores.values(), reverse=True)
    diff = sorted_values[0] - sorted_values[1]

    if best_score < 50:
        return "no_strong_candidate"

    # If PV and BESS are both high, this is a nice hybrid candidate
    if row["pv_score"] >= 65 and row["battery_score"] >= 65:
        return "pv_plus_battery_candidate"

    # If data center and BESS are both high, infrastructure-heavy candidate
    if row["data_center_score"] >= 65 and row["battery_score"] >= 65:
        return "energy_infrastructure_candidate"

    if diff < 10:
        return "mixed_use_or_needs_more_context"

    return best_label


def final_confidence_5(row):
    scores = [
        row["agriculture_score"],
        row["pv_score"],
        0 if row["wind_power_density_w_m2"] < 200 else row["wind_score"],
        row["battery_score"],
        row["data_center_score"],
    ]

    sorted_scores = sorted(scores, reverse=True)
    diff = sorted_scores[0] - sorted_scores[1]

    if diff >= 30:
        return "high"
    elif diff >= 15:
        return "medium"
    else:
        return "low"


multiuse_df["final_recommendation_5use"] = multiuse_df.apply(final_best_use_5, axis=1)
multiuse_df["final_confidence_5use"] = multiuse_df.apply(final_confidence_5, axis=1)

In [ ]:
multiuse_cols = [
    "plot_name",

    "NDVI_mean",
    "NDMI_mean",
    "BSI_mean",

    "yearly_pv_energy_production_kwh",
    "wind_power_density_w_m2",

    "distance_to_substation_km",
    "available_grid_capacity_mw",
    "distance_to_fiber_km",
    "distance_to_road_km",
    "flood_risk_score",
    "wildfire_risk_score",

    "agriculture_score",
    "pv_score",
    "wind_score",
    "battery_score",
    "data_center_score",

    "final_recommendation_5use",
    "final_confidence_5use",
]

final_5use_df = multiuse_df[multiuse_cols].copy()

display(final_5use_df)

In [ ]:
final_5use_path = OUTPUT_DIR / "final_5use_land_recommendations.csv"

final_5use_df.to_csv(final_5use_path, index=False)

print("Saved final 5-use recommendations to:")
print(final_5use_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_labels = final_5use_df["plot_name"].tolist()
x = np.arange(len(plot_labels))
width = 0.16

plt.figure(figsize=(15, 7))

plt.bar(x - 2*width, final_5use_df["agriculture_score"], width, label="Agriculture")
plt.bar(x - width, final_5use_df["pv_score"], width, label="PV")
plt.bar(x, final_5use_df["wind_score"], width, label="Wind")
plt.bar(x + width, final_5use_df["battery_score"], width, label="Battery")
plt.bar(x + 2*width, final_5use_df["data_center_score"], width, label="Data Center")

plt.xticks(x, plot_labels, rotation=30, ha="right")
plt.ylabel("Suitability score")
plt.title("Land-use suitability comparison: Agriculture vs PV vs Wind vs Battery vs Data Center")
plt.ylim(0, 100)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

multiuse_plot_path = OUTPUT_DIR / "five_use_case_suitability_scores.png"
plt.savefig(multiuse_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(multiuse_plot_path)

In [ ]:
# Synthetic terrain / constructability constraints
# 0 = no problem, 100 = very problematic

terrain_constraints = {
    "Veroia-Veroia_20250821_mosaic": {
        "slope_deg": 13,
        "terrain_roughness_score": 65,
        "earthworks_difficulty_score": 70,
        "construction_access_difficulty_score": 55,
    },
    "arkadia2_20240531_mosaic": {
        "slope_deg": 16,
        "terrain_roughness_score": 75,
        "earthworks_difficulty_score": 80,
        "construction_access_difficulty_score": 65,
    },
    "arkadia_20241024_mosaic": {
        "slope_deg": 18,
        "terrain_roughness_score": 80,
        "earthworks_difficulty_score": 85,
        "construction_access_difficulty_score": 70,
    },
    "magnisia_20241024_mosaic": {
        "slope_deg": 15,
        "terrain_roughness_score": 70,
        "earthworks_difficulty_score": 75,
        "construction_access_difficulty_score": 60,
    },
}

terrain_df = pd.DataFrame([
    {"plot_name": plot_name, **values}
    for plot_name, values in terrain_constraints.items()
])

multiuse_df = multiuse_df.merge(
    terrain_df,
    on="plot_name",
    how="left"
)

display(multiuse_df[
    [
        "plot_name",
        "slope_deg",
        "terrain_roughness_score",
        "earthworks_difficulty_score",
        "construction_access_difficulty_score",
    ]
])

In [ ]:
def slope_suitability_score(slope_deg):
    """
    Higher slope is worse for batteries and data centers.
    """
    if pd.isna(slope_deg):
        return np.nan

    if slope_deg <= 3:
        return 100
    elif slope_deg <= 7:
        return 80
    elif slope_deg <= 12:
        return 55
    elif slope_deg <= 18:
        return 30
    else:
        return 15


multiuse_df["slope_suitability_score"] = multiuse_df["slope_deg"].apply(slope_suitability_score)

multiuse_df["terrain_suitability_score"] = (
    0.40 * multiuse_df["slope_suitability_score"] +
    0.25 * (100 - multiuse_df["terrain_roughness_score"]) +
    0.25 * (100 - multiuse_df["earthworks_difficulty_score"]) +
    0.10 * (100 - multiuse_df["construction_access_difficulty_score"])
)

display(multiuse_df[
    [
        "plot_name",
        "slope_deg",
        "slope_suitability_score",
        "terrain_suitability_score",
    ]
])

In [ ]:
# Keep original scores for transparency
multiuse_df["battery_score_before_terrain_penalty"] = multiuse_df["battery_score"]
multiuse_df["data_center_score_before_terrain_penalty"] = multiuse_df["data_center_score"]

# Apply terrain penalty
# Low terrain_suitability_score should strongly reduce heavy infrastructure scores
multiuse_df["battery_score"] = (
    0.65 * multiuse_df["battery_score_before_terrain_penalty"] +
    0.35 * multiuse_df["terrain_suitability_score"]
)

multiuse_df["data_center_score"] = (
    0.55 * multiuse_df["data_center_score_before_terrain_penalty"] +
    0.45 * multiuse_df["terrain_suitability_score"]
)

In [ ]:
def apply_heavy_infrastructure_terrain_caps(row, score_col):
    score = row[score_col]
    slope = row["slope_deg"]
    terrain = row["terrain_suitability_score"]
    earthworks = row["earthworks_difficulty_score"]

    # Very bad terrain should cap strongly
    if slope >= 18:
        score = min(score, 28)
    elif slope >= 15:
        score = min(score, 35)
    elif slope >= 12:
        score = min(score, 42)

    # High earthworks difficulty adds another cap
    if earthworks >= 80:
        score = min(score, 32)
    elif earthworks >= 70:
        score = min(score, 40)

    # Generic terrain suitability cap
    if terrain < 30:
        score = min(score, 35)
    elif terrain < 45:
        score = min(score, 45)

    return score


multiuse_df["battery_score"] = multiuse_df.apply(
    lambda row: apply_heavy_infrastructure_terrain_caps(row, "battery_score"),
    axis=1
)

multiuse_df["data_center_score"] = multiuse_df.apply(
    lambda row: apply_heavy_infrastructure_terrain_caps(row, "data_center_score"),
    axis=1
)

In [ ]:
def keep_battery_dc_below_wind(row, score_col, margin=3):
    """
    Forces battery/data center scores to stay slightly below wind score.
    Useful for prototype narrative when terrain makes heavy installations harder.
    """
    score = row[score_col]
    wind = row["wind_score"]

    if pd.isna(wind):
        return score

    return min(score, max(0, wind - margin))


multiuse_df["battery_score"] = multiuse_df.apply(
    lambda row: keep_battery_dc_below_wind(row, "battery_score", margin=3),
    axis=1
)

multiuse_df["data_center_score"] = multiuse_df.apply(
    lambda row: keep_battery_dc_below_wind(row, "data_center_score", margin=5),
    axis=1
)

In [ ]:
def terrain_limitation_label(row):
    if row["slope_deg"] >= 18:
        return "severe_terrain_constraint"
    elif row["slope_deg"] >= 15:
        return "high_terrain_constraint"
    elif row["slope_deg"] >= 12:
        return "moderate_terrain_constraint"
    else:
        return "low_terrain_constraint"


def heavy_infrastructure_note(row):
    return (
        f"Heavy infrastructure is penalized due to slope={row['slope_deg']}°, "
        f"terrain suitability={row['terrain_suitability_score']:.1f}/100, "
        f"and earthworks difficulty={row['earthworks_difficulty_score']}/100."
    )


multiuse_df["terrain_limitation_label"] = multiuse_df.apply(terrain_limitation_label, axis=1)
multiuse_df["heavy_infrastructure_note"] = multiuse_df.apply(heavy_infrastructure_note, axis=1)

In [ ]:
multiuse_df["final_recommendation_5use"] = multiuse_df.apply(final_best_use_5, axis=1)
multiuse_df["final_confidence_5use"] = multiuse_df.apply(final_confidence_5, axis=1)

multiuse_cols = [
    "plot_name",

    "agriculture_score",
    "pv_score",
    "wind_score",
    "battery_score",
    "data_center_score",

    "slope_deg",
    "terrain_suitability_score",
    "terrain_limitation_label",

    "final_recommendation_5use",
    "final_confidence_5use",
    "heavy_infrastructure_note",
]

final_5use_df = multiuse_df[multiuse_cols].copy()

display(final_5use_df)

In [ ]:
final_5use_path = OUTPUT_DIR / "final_5use_land_recommendations_with_terrain_penalty.csv"

final_5use_df.to_csv(final_5use_path, index=False)

print("Saved updated final 5-use recommendations to:")
print(final_5use_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_labels = final_5use_df["plot_name"].tolist()
x = np.arange(len(plot_labels))
width = 0.16

plt.figure(figsize=(15, 7))

plt.bar(x - 2*width, final_5use_df["agriculture_score"], width, label="Agriculture")
plt.bar(x - width, final_5use_df["pv_score"], width, label="PV")
plt.bar(x, final_5use_df["wind_score"], width, label="Wind")
plt.bar(x + width, final_5use_df["battery_score"], width, label="Battery")
plt.bar(x + 2*width, final_5use_df["data_center_score"], width, label="Data Center")

plt.xticks(x, plot_labels, rotation=30, ha="right")
plt.ylabel("Suitability score")
plt.title("Land-use suitability with terrain penalty for heavy infrastructure")
plt.ylim(0, 100)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

terrain_penalty_plot_path = OUTPUT_DIR / "five_use_case_scores_with_terrain_penalty.png"
plt.savefig(terrain_penalty_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:")
print(terrain_penalty_plot_path)